In [1]:
# Install ipython-sql if not already installed
# !pip install ipython-sql

In [2]:
import sqlite3
import sql
import pandas as pd
import numpy as np

# Connect to a database (creates the database file if it doesn't exist)
cnn = sqlite3.connect('northwind1.db')

In [3]:
import sql

# Load the SQL extension
%load_ext sql

# Connect to a SQLite database
%sql sqlite:///northwind1.db

#display style of SQL query results in Jupyter Notebook
%config SqlMagic.style = '_DEPRECATED_DEFAULT'

In [4]:
%%sql

Drop table Monthly_Sales;

 * sqlite:///northwind1.db
Done.


[]

In [5]:
%%sql

-- Create the table
CREATE TABLE IF NOT EXISTS Monthly_Sales (
    Number INT,
    Month VARCHAR(20) PRIMARY KEY,
    Revenue INT
);


-- Insert rows from Jan to July
INSERT OR IGNORE INTO Monthly_Sales (Number, Month, Revenue) 
VALUES 
(1, 'Jan', 100),
(2, 'Feb', 300),
(3, 'Mar', 200),
(4, 'Apr', 500),
(5, 'May', 400),
(6, 'Jun', 600),
(7, 'Jul', 700);

 * sqlite:///northwind1.db
Done.
Done.


[]

In [6]:
%%sql

SELECT * FROM Monthly_Sales;

 * sqlite:///northwind1.db
Done.


Number,Month,Revenue
1,Jan,100
2,Feb,300
3,Mar,200
4,Apr,500
5,May,400
6,Jun,600
7,Jul,700


# 1️⃣ Cumulative Sum (From Start till Current Row)

In [7]:
%%sql

-- 👉 Frame: Start of partition → Current Row


SELECT Number, Month, Revenue,
       SUM(Revenue) OVER (
         ORDER BY Number
         ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
       ) AS cum_sum
FROM Monthly_Sales;

 * sqlite:///northwind1.db
Done.


Number,Month,Revenue,cum_sum
1,Jan,100,100
2,Feb,300,400
3,Mar,200,600
4,Apr,500,1100
5,May,400,1500
6,Jun,600,2100
7,Jul,700,2800


# 2️⃣ Full Total (From Start till End)

In [8]:
%%sql

-- 👉 Frame: Entire partition

SELECT Number, Month, Revenue,
       SUM(Revenue) OVER (
         ORDER BY Number
         ROWS BETWEEN UNBOUNDED PRECEDING AND UNBOUNDED FOLLOWING
       ) AS total_sum
FROM Monthly_Sales;


 * sqlite:///northwind1.db
Done.


Number,Month,Revenue,total_sum
1,Jan,100,2800
2,Feb,300,2800
3,Mar,200,2800
4,Apr,500,2800
5,May,400,2800
6,Jun,600,2800
7,Jul,700,2800


#   3️⃣ Moving Window of 1 Previous + Current Row

In [11]:
%%sql


-- 👉 Frame: Current + 1 before

SELECT Number, Month, Revenue,
       SUM(Revenue) OVER (
         ORDER BY Number
         ROWS BETWEEN 1 PRECEDING AND CURRENT ROW
       ) AS moving_sum
FROM Monthly_Sales;

 * sqlite:///northwind1.db
Done.


Number,Month,Revenue,moving_sum
1,Jan,100,100
2,Feb,300,400
3,Mar,200,500
4,Apr,500,700
5,May,400,900
6,Jun,600,1000
7,Jul,700,1300


# 4️⃣ Moving Window of Current + 1 Following Row

In [12]:
%%sql

-- 👉 Frame: Current + 1 after

SELECT Number, Month, Revenue,
       SUM(Revenue) OVER (
         ORDER BY Number
         ROWS BETWEEN CURRENT ROW AND 1 FOLLOWING
       ) AS moving_sum
FROM Monthly_Sales;

 * sqlite:///northwind1.db
Done.


Number,Month,Revenue,moving_sum
1,Jan,100,400
2,Feb,300,500
3,Mar,200,700
4,Apr,500,900
5,May,400,1000
6,Jun,600,1300
7,Jul,700,700


# 5️⃣ Centered Moving Window (1 Before, Current, 1 After)

In [13]:
%%sql

-- 👉 Frame: One before + Current + One after

SELECT Number, Month, Revenue,
       SUM(Revenue) OVER (
         ORDER BY Number
         ROWS BETWEEN 1 PRECEDING AND 1 FOLLOWING
       ) AS moving_avg
FROM Monthly_Sales;

 * sqlite:///northwind1.db
Done.


Number,Month,Revenue,moving_avg
1,Jan,100,400
2,Feb,300,600
3,Mar,200,1000
4,Apr,500,1100
5,May,400,1500
6,Jun,600,1700
7,Jul,700,1300


# 6️⃣ Future Cumulative (Current → End)

In [14]:
%%sql

-- 👉 Frame: Current till end

SELECT Number, Month, Revenue,
       SUM(Revenue) OVER (
         ORDER BY Number
         ROWS BETWEEN CURRENT ROW AND UNBOUNDED FOLLOWING
       ) AS future_sum
FROM Monthly_Sales;

 * sqlite:///northwind1.db
Done.


Number,Month,Revenue,future_sum
1,Jan,100,2800
2,Feb,300,2700
3,Mar,200,2400
4,Apr,500,2200
5,May,400,1700
6,Jun,600,1300
7,Jul,700,700


# Notebook Ends Here.

# 8️⃣ Combining Multiple Window Aggregates

In [ ]:
%%sql
SELECT 
    ProductID,
    ProductName,
    CategoryID,
    Price,
    SUM(Price) OVER () AS total_price_all_products,
    COUNT(*) OVER (PARTITION BY CategoryID) AS product_count,
    SUM(Price) OVER (PARTITION BY CategoryID) AS total_price,
    AVG(Price) OVER (PARTITION BY CategoryID) AS avg_price,
    MIN(Price) OVER (PARTITION BY CategoryID) AS min_price,
    MAX(Price) OVER (PARTITION BY CategoryID) AS max_price
FROM Products;

In [ ]:
%%sql
SELECT 
    CategoryID,
    COUNT(*) AS product_count,
    SUM(Price) AS total_price,
    AVG(Price) AS avg_price,
    MIN(Price) AS min_price,
    MAX(Price) AS max_price
FROM Products
GROUP BY CategoryID;

# 9️⃣ COUNT() with Running Count (ROW-Like Behavior)

In [ ]:
%%sql
SELECT 
    ProductID,
    ProductName,
    CategoryID,
    Price,
    COUNT(*) OVER (
        PARTITION BY CategoryID
        ORDER BY ProductID
        ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
    ) AS running_product_count
FROM Products;

GROUP BY Equivalent

❌ Like cumulative sums, running counts cannot be done with plain GROUP BY.
We can use correlated subquery:

In [ ]:
%%sql
SELECT 
    p1.ProductID,
    p1.ProductName,
    p1.CategoryID,
    p1.Price,
    (
        SELECT COUNT(*)
        FROM Products p2
        WHERE p2.CategoryID = p1.CategoryID
          AND p2.ProductID <= p1.ProductID
    ) AS running_product_count
FROM Products p1
ORDER BY p1.CategoryID, p1.ProductID;

# Count the number of customers per country

In [ ]:
%%sql

SELECT 

Country, 

COUNT(*) AS Total_Customers

FROM Customers

GROUP BY Country;

In [ ]:
%%sql

SELECT 

COUNT(*) AS Total_Customers

FROM Customers;

In [ ]:
%%sql

SELECT 
Country, 
COUNT(*) AS Total_Customers
FROM Customers
GROUP BY Country
ORDER BY Total_Customers
desc;

# Get the average price of the product in each category

In [ ]:
%%sql

SELECT *
FROM Products
Limit 5;

In [ ]:
%%sql

SELECT AVG(Price) AS AvgPrice 
FROM Products;

In [ ]:
%%sql

SELECT AVG(Price) AS AvgPrice 
FROM Products
group by CategoryID;

In [ ]:
%%sql

SELECT CategoryID, AVG(Price) AS AvgPrice 
FROM Products
group by CategoryID;

In [ ]:
%%sql

select * FROM categories;

In [ ]:
%%sql

SELECT CategoryID, AVG(Price) AS AvgPrice 
FROM Products
group by CategoryID
order by AVG(Price);

In [ ]:
%%sql

SELECT CategoryID, AVG(Price) AS AvgPrice 
FROM Products
group by CategoryID
order by AVG(Price);

## repeate same for sum of price within each category

# Get Minimum and Maxmium Price per Category

In [ ]:
%%sql

SELECT CategoryID, 
MAX(Price) AS Max_Price, 
MIN(Price) AS Min_Price

FROM Products
GROUP BY CategoryID;

In [ ]:
%%sql

select * FROM categories;

In [ ]:
%%sql

SELECT 
    c.CategoryName,
    p.CategoryID, 
    MAX(p.Price) AS Max_Price, 
    MIN(p.Price) AS Min_Price
FROM Products p
JOIN Categories c 
    ON p.CategoryID = c.CategoryID
GROUP BY 
    c.CategoryName, 
    p.CategoryID;

In [ ]:
%%sql

SELECT country,
city,
count(*) AS Total_customers_per_country_per_city
From Customers

group by country, city;

In [ ]:
%%sql

SELECT country, city, count(*) AS total_cust

From Customers

group by country, city

having count(*) > 2

order by count(*) asc;

# 🚨 Common Mistakes
## ❌ Using columns in SELECT that are not in GROUP BY or aggregate functions.

## ❌ Using WHERE instead of HAVING for aggregated values.

# ✅ Best Practices

### Always use aggregate functions with columns not in the GROUP BY clause.

### Use HAVING to filter aggregated results.

### Use aliases to improve readability of output.

In [ ]:
%%sql

SELECT * 
FROM Suppliers
Limit 5;

# Get all suppliers and customers from London. (Using Multiple CTEs)

In [ ]:
%%sql

WITH LondonCustomers AS (
    SELECT CustomerID, ContactName, City FROM Customers WHERE City = 'London'
),
LondonSuppliers AS (
    SELECT SupplierID, ContactName, City FROM  WHERE City =Suppliers 'London'
)
SELECT * FROM LondonCustomers
UNION ALL
SELECT * FROM LondonSuppliers;

In [ ]:
%%sql

WITH LondonCustomers AS (
    SELECT 
        CustomerID AS ID, 
        ContactName, 
        City, 
        'Customer' AS Type
    FROM Customers 
    WHERE City = 'London'
),
LondonSuppliers AS (
    SELECT 
        SupplierID AS ID, 
        ContactName, 
        City, 
        'Supplier' AS Type
    FROM Suppliers 
    WHERE City = 'London'
)
SELECT * FROM LondonCustomers
UNION ALL
SELECT * FROM LondonSuppliers;

In [ ]:
%%sql

SELECT * FROM Products
Limit 10;

In [ ]:
%%sql

WITH AvgPriceCTE AS (
    SELECT AVG(Price) AS AvgPrice FROM Products
)
SELECT *
FROM Products
WHERE Price > (SELECT AvgPrice FROM AvgPriceCTE);

# 📌 This gives you the number of products in each category.

In [ ]:
%%sql

WITH CategoryCounts AS (
    SELECT CategoryID, COUNT(*) AS TotalProducts
    FROM Products
    GROUP BY CategoryID
)
SELECT *
FROM CategoryCounts;

# CTE to Identify Duplicate Prices
📌 Finds all products that share the same price as others (duplicates).

In [ ]:
%%sql

WITH PriceCounts AS (
    SELECT Price, COUNT(*) AS Occurrences
    FROM Products
    GROUP BY Price
    HAVING COUNT(*) > 1
)
SELECT *
FROM Products
WHERE Price IN (SELECT Price FROM PriceCounts) order by price;

# CTE to Get the Cheapest Product Per Category

In [ ]:
%%sql

SELECT CategoryID, MIN(Price) AS MinPrice
    FROM Products
    GROUP BY CategoryID;

In [ ]:
%%sql

WITH MinPrices AS (
    SELECT CategoryID, MIN(Price) AS MinPrice
    FROM Products
    GROUP BY CategoryID
)
SELECT *
FROM Products
WHERE (CategoryID, Price) IN (
    SELECT CategoryID, MinPrice FROM MinPrices
) order by CategoryID;

# Get the employees who handle more than 2 territories.

In [ ]:
%%sql

WITH TerritoryCount AS (
    SELECT EmployeeID, COUNT(*) AS TotalTerritories
    FROM EmployeeTerritories
    GROUP BY EmployeeID
)
SELECT E.FirstName, E.LastName, T.TotalTerritories
FROM Employees E
JOIN TerritoryCount T ON E.EmployeeID = T.EmployeeID
WHERE T.TotalTerritories > 2;

# 📌 Returns the most expensive product in each category.

In [ ]:
%%sql

WITH RankedCategory AS (
    SELECT ProductID, ProductName, CategoryID, Price,
           RANK() OVER (PARTITION BY CategoryID ORDER BY Price DESC) AS RankInCategory
    FROM Products
)
SELECT *
FROM RankedCategory
WHERE RankInCategory = 1;

# 📌 Filters category 3 products, then returns the top 3 most expensive among them.

In [ ]:
%%sql

WITH Filtered AS (
    SELECT * FROM Products WHERE CategoryID = 3
),
Ranked AS (
    SELECT ProductID, ProductName, Price,
           DENSE_RANK() OVER (ORDER BY Price DESC) AS PriceRank
    FROM Filtered
)
SELECT * FROM Ranked WHERE PriceRank <= 3;

In [ ]:
%%sql

SELECT * FROM Products
ORDER BY ProductName;

# 📌 Categorizes products into 'Low', 'Medium', and 'High' price brackets.

In [ ]:
%%sql

WITH PriceBuckets AS (
    SELECT ProductID, ProductName, Price,
           CASE 
               WHEN Price < 10 THEN 'Low'
               WHEN Price BETWEEN 10 AND 30 THEN 'Medium'
               ELSE 'High'
           END AS PriceCategory
    FROM Products
)
SELECT *
FROM PriceBuckets
WHERE PriceCategory = 'Low';

# Recursive CTEs

In [ ]:
%%sql

WITH RECURSIVE FactorialCTE(n, fact) AS (
    SELECT 1, 1
    UNION ALL
    SELECT n + 1, (n + 1) * fact
    FROM FactorialCTE
    WHERE n < 5
)
SELECT * FROM FactorialCTE;

# Return all customers that starts with "E" and are at least 3 characters in length

In [ ]:
%%sql

SELECT * FROM Customers
WHERE CustomerName LIKE 'E__%';

# END..... Practice Below...

In [ ]:
%%sql

SELECT * 
FROM Customers
WHERE country = 'Brazil';

In [ ]:
%%sql

SELECT * 
FROM Customers
WHERE CustomerID=3;

In [ ]:
%%sql

SELECT * FROM Products;

In [ ]:
%%sql

SELECT *
FROM Products
WHERE Price > 40;

In [ ]:
%%sql

SELECT * 
FROM Products
WHERE Price < 40;

In [ ]:
%%sql

SELECT * 
FROM Products
WHERE Price >= 30;

In [ ]:
%%sql

SELECT * 
FROM Products
WHERE Price <= 30;

In [ ]:
%%sql

SELECT * FROM Products
WHERE Price <> 30;

In [ ]:
%%sql

SELECT * 
FROM Products
WHERE Price BETWEEN 30 AND 40;

In [ ]:
%%sql

SELECT * 
FROM Customers
WHERE City IN ('Paris','London');

In [ ]:
%%sql



In [ ]:
%%sql



In [ ]:
%%sql



In [ ]:
%%sql



In [ ]:
%%sql

